### Setup

In [1]:
from pathlib import Path
import time
import pickle
import pandas as pd
from tqdm import tqdm
from fastf1.ergast import Ergast

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW = PROJECT_ROOT / "data" / "raw"
CACHE = RAW / "api_cache"
RACE_CACHE = CACHE / "race_results"
QUALI_CACHE = CACHE / "qualifying_results"

RAW.mkdir(parents=True, exist_ok=True)
RACE_CACHE.mkdir(parents=True, exist_ok=True)
QUALI_CACHE.mkdir(parents=True, exist_ok=True)

START_YEAR = 2010
END_YEAR = 2025

REQUEST_SLEEP = 2
RETRY_SLEEP = 20

ergast = Ergast(result_type="pandas", auto_cast=True, limit=2000)

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW)
print("Cache folder:", CACHE)

Project root: /Users/lukeweeklund/F1 Project
Raw folder: /Users/lukeweeklund/F1 Project/data/raw
Cache folder: /Users/lukeweeklund/F1 Project/data/raw/api_cache


### Helper functions

In [2]:
def safe_call(func, retries=4, wait=RETRY_SLEEP, **kwargs):
    for attempt in range(retries):
        try:
            return func(**kwargs)
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt == retries - 1:
                raise
            time.sleep(wait * (attempt + 1))


def cache_path(folder, prefix, season, round_number):
    return folder / f"{prefix}_{int(season)}_{int(round_number):02d}.pkl"


def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def flatten_response(response, label):
    frames = []

    for i, df in enumerate(response.content):
        if df.empty:
            continue

        meta = response.description.iloc[i].to_dict()
        temp = df.copy()

        for col in [
            "season", "round", "raceName", "date",
            "circuitId", "circuitName", "locality",
            "country", "lat", "long"
        ]:
            temp[col] = meta.get(col)

        temp["data_type"] = label
        frames.append(temp)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True, sort=False)


def clean_driver_race_table(df):
    df = df.copy()

    df["season"] = pd.to_numeric(df["season"], errors="coerce")
    df["round"] = pd.to_numeric(df["round"], errors="coerce")

    df = df.dropna(subset=["season", "round", "driverId"]).copy()

    df["season"] = df["season"].astype(int)
    df["round"] = df["round"].astype(int)

    df = df.drop_duplicates(
        subset=["season", "round", "driverId"],
        keep="last"
    ).reset_index(drop=True)

    return df

### Pull schedule

In [3]:
schedule_frames = []
failed_schedules = []

for year in tqdm(range(START_YEAR, END_YEAR + 1)):
    try:
        schedule = safe_call(ergast.get_race_schedule, season=year)
        schedule["season"] = year
        schedule_frames.append(schedule)
        time.sleep(REQUEST_SLEEP)

    except Exception as e:
        failed_schedules.append((year, str(e)))
        print(f"Schedule failed {year}: {e}")
        time.sleep(RETRY_SLEEP)

schedule_df = pd.concat(schedule_frames, ignore_index=True, sort=False)

schedule_df["season"] = pd.to_numeric(schedule_df["season"], errors="coerce")
schedule_df["round"] = pd.to_numeric(schedule_df["round"], errors="coerce")

schedule_df = schedule_df.dropna(subset=["season", "round"]).copy()

schedule_df["season"] = schedule_df["season"].astype(int)
schedule_df["round"] = schedule_df["round"].astype(int)

schedule_df = schedule_df.drop_duplicates(
    subset=["season", "round"],
    keep="last"
).sort_values(["season", "round"]).reset_index(drop=True)

schedule_races = list(
    schedule_df[["season", "round"]]
    .itertuples(index=False, name=None)
)

print("Schedule:", schedule_df.shape)
print("Unique schedule races:", len(schedule_races))
print("Failed schedules:", len(failed_schedules))

req         WARNING 	DEFAULT CACHE ENABLED! (15.08 MB) /Users/lukeweeklund/Library/Caches/fastf1
100%|███████████████████████████████████████████| 16/16 [00:32<00:00,  2.02s/it]

Schedule: (329, 23)
Unique schedule races: 329
Failed schedules: 0


### Download missing race-result responses to cache

In [4]:
failed_race_results = []

missing_race_results = []

for season, round_number in schedule_races:
    path = cache_path(RACE_CACHE, "race_results", season, round_number)

    if not path.exists():
        missing_race_results.append((season, round_number))

print("Race-result cache files already present:", len(schedule_races) - len(missing_race_results))
print("Race-result cache files missing:", len(missing_race_results))

for season, round_number in tqdm(missing_race_results):
    try:
        response = safe_call(
            ergast.get_race_results,
            season=int(season),
            round=int(round_number)
        )

        path = cache_path(RACE_CACHE, "race_results", season, round_number)
        save_pickle(response, path)

        time.sleep(REQUEST_SLEEP)

    except Exception as e:
        failed_race_results.append((season, round_number, str(e)))
        print(f"Race results failed {season} round {round_number}: {e}")
        time.sleep(RETRY_SLEEP)

print("Failed race-result downloads:", len(failed_race_results))

Race-result cache files already present: 0
Race-result cache files missing: 329


100%|█████████████████████████████████████████| 329/329 [12:57<00:00,  2.36s/it]

Failed race-result downloads: 0


### Download missing qualifying responses to cache

In [6]:
failed_qualifying = []

missing_qualifying = []

for season, round_number in schedule_races:
    path = cache_path(QUALI_CACHE, "qualifying_results", season, round_number)

    if not path.exists():
        missing_qualifying.append((season, round_number))

print("Qualifying cache files already present:", len(schedule_races) - len(missing_qualifying))
print("Qualifying cache files missing:", len(missing_qualifying))

for season, round_number in tqdm(missing_qualifying):
    try:
        response = safe_call(
            ergast.get_qualifying_results,
            season=int(season),
            round=int(round_number)
        )

        path = cache_path(QUALI_CACHE, "qualifying_results", season, round_number)
        save_pickle(response, path)

        time.sleep(REQUEST_SLEEP)

    except Exception as e:
        failed_qualifying.append((season, round_number, str(e)))
        print(f"Qualifying failed {season} round {round_number}: {e}")
        time.sleep(RETRY_SLEEP)

print("Failed qualifying downloads:", len(failed_qualifying))

Qualifying cache files already present: 286
Qualifying cache files missing: 43


100%|███████████████████████████████████████████| 43/43 [01:47<00:00,  2.51s/it]

Failed qualifying downloads: 0


### Build race results CSV from cache

In [7]:
race_frames = []

for season, round_number in tqdm(schedule_races):
    path = cache_path(RACE_CACHE, "race_results", season, round_number)

    if not path.exists():
        continue

    response = load_pickle(path)
    race_df = flatten_response(response, "race_results")

    if not race_df.empty:
        race_frames.append(race_df)

race_results_df = pd.concat(race_frames, ignore_index=True, sort=False)
race_results_df = clean_driver_race_table(race_results_df)

race_results_df.to_csv(RAW / "race_results.csv", index=False)

print("Race results:", race_results_df.shape)
print("Unique race-result races:", race_results_df[["season", "round"]].drop_duplicates().shape[0])
print("Duplicate driver-race rows:", race_results_df.duplicated(["season", "round", "driverId"]).sum())

100%|████████████████████████████████████████| 329/329 [00:00<00:00, 951.23it/s]


Race results: (6911, 37)
Unique race-result races: 329
Duplicate driver-race rows: 0


### Build qualifying CSV from cache

In [8]:
qualifying_frames = []

for season, round_number in tqdm(schedule_races):
    path = cache_path(QUALI_CACHE, "qualifying_results", season, round_number)

    if not path.exists():
        continue

    response = load_pickle(path)
    quali_df = flatten_response(response, "qualifying_results")

    if not quali_df.empty:
        qualifying_frames.append(quali_df)

qualifying_results_df = pd.concat(qualifying_frames, ignore_index=True, sort=False)
qualifying_results_df = clean_driver_race_table(qualifying_results_df)

qualifying_results_df.to_csv(RAW / "qualifying_results.csv", index=False)

print("Qualifying:", qualifying_results_df.shape)
print("Unique qualifying races:", qualifying_results_df[["season", "round"]].drop_duplicates().shape[0])
print("Duplicate qualifying driver-race rows:", qualifying_results_df.duplicated(["season", "round", "driverId"]).sum())

100%|████████████████████████████████████████| 329/329 [00:00<00:00, 871.80it/s]


Qualifying: (6891, 28)
Unique qualifying races: 329
Duplicate qualifying driver-race rows: 0


### Circuit lookup

In [9]:
circuit_cols = [
    "season", "round", "raceName", "circuitId", "circuitName",
    "locality", "country", "lat", "long"
]

available_circuit_cols = [
    col for col in circuit_cols
    if col in schedule_df.columns
]

circuit_lookup_df = (
    schedule_df[available_circuit_cols]
    .drop_duplicates()
    .sort_values(["season", "round"])
    .reset_index(drop=True)
)

circuit_lookup_df.to_csv(RAW / "circuit_lookup.csv", index=False)
schedule_df.to_csv(RAW / "race_schedule.csv", index=False)

print("Circuit lookup:", circuit_lookup_df.shape)
print("Unique circuits:", circuit_lookup_df["circuitId"].nunique())

Circuit lookup: (329, 9)
Unique circuits: 35


### Validation

In [10]:
print("=" * 80)
print("NOTEBOOK 1 VALIDATION — RAW DATA PULL")
print("=" * 80)

schedule_race_count = schedule_df[["season", "round"]].drop_duplicates().shape[0]
race_result_race_count = race_results_df[["season", "round"]].drop_duplicates().shape[0]
qualifying_race_count = qualifying_results_df[["season", "round"]].drop_duplicates().shape[0]

drivers_per_race = race_results_df.groupby(["season", "round"]).size()

race_completed = set(
    zip(race_results_df["season"], race_results_df["round"])
)

quali_completed = set(
    zip(qualifying_results_df["season"], qualifying_results_df["round"])
)

missing_race_results_final = [
    x for x in schedule_races
    if x not in race_completed
]

missing_qualifying_final = [
    x for x in schedule_races
    if x not in quali_completed
]

print("\nSHAPES")
print("-" * 80)
print("Schedule:", schedule_df.shape)
print("Race results:", race_results_df.shape)
print("Qualifying:", qualifying_results_df.shape)
print("Circuit lookup:", circuit_lookup_df.shape)

print("\nUNIQUE RACES")
print("-" * 80)
print("Schedule races:", schedule_race_count)
print("Race result races:", race_result_race_count)
print("Qualifying races:", qualifying_race_count)

print("\nROW EXPECTATIONS")
print("-" * 80)
print("Average drivers per race:", drivers_per_race.mean())
print("Min drivers in a race:", drivers_per_race.min())
print("Max drivers in a race:", drivers_per_race.max())

print("\nCACHE COVERAGE")
print("-" * 80)
print("Race-result cache files:", len(list(RACE_CACHE.glob('race_results_*.pkl'))))
print("Qualifying cache files:", len(list(QUALI_CACHE.glob('qualifying_results_*.pkl'))))

print("\nMISSING COVERAGE")
print("-" * 80)
print("Missing race-result races:", len(missing_race_results_final))
print("Missing qualifying races:", len(missing_qualifying_final))

if missing_race_results_final:
    display(pd.DataFrame(missing_race_results_final, columns=["season", "round"]).head(30))

if missing_qualifying_final:
    display(pd.DataFrame(missing_qualifying_final, columns=["season", "round"]).head(30))

print("\nFAILED THIS RUN")
print("-" * 80)
print("Failed schedules:", len(failed_schedules))
print("Failed race-result downloads:", len(failed_race_results))
print("Failed qualifying downloads:", len(failed_qualifying))

issues = []

if failed_schedules:
    issues.append("Schedule pull had failures.")

if failed_race_results:
    issues.append("Race result pull had failures.")

if schedule_race_count != race_result_race_count:
    issues.append("Race result coverage does not match schedule coverage.")

if race_results_df.duplicated(["season", "round", "driverId"]).sum() != 0:
    issues.append("Race results contain duplicate driver-race rows.")

if drivers_per_race.mean() < 15:
    issues.append("Average drivers per race looks too low.")

if qualifying_race_count < schedule_race_count * 0.90:
    issues.append("Qualifying coverage is unexpectedly low.")

if qualifying_results_df.duplicated(["season", "round", "driverId"]).sum() != 0:
    issues.append("Qualifying contains duplicate driver-race rows.")

if issues:
    print("\nVERDICT: REVIEW NEEDED")
    for issue in issues:
        print("-", issue)
else:
    print("\nVERDICT: PASS — raw data pull looks complete.")

NOTEBOOK 1 VALIDATION — RAW DATA PULL

SHAPES
--------------------------------------------------------------------------------
Schedule: (329, 23)
Race results: (6911, 37)
Qualifying: (6891, 28)
Circuit lookup: (329, 9)

UNIQUE RACES
--------------------------------------------------------------------------------
Schedule races: 329
Race result races: 329
Qualifying races: 329

ROW EXPECTATIONS
--------------------------------------------------------------------------------
Average drivers per race: 21.00607902735562
Min drivers in a race: 18
Max drivers in a race: 24

CACHE COVERAGE
--------------------------------------------------------------------------------
Race-result cache files: 329
Qualifying cache files: 329

MISSING COVERAGE
--------------------------------------------------------------------------------
Missing race-result races: 0
Missing qualifying races: 0

FAILED THIS RUN
--------------------------------------------------------------------------------
Failed schedules:

### Supercheck

In [14]:
# ==============================================================================
# SUPERCHECK — NOTEBOOK 1 RAW DATA INTEGRITY
# ==============================================================================

print("=" * 90)
print("SUPERCHECK — NOTEBOOK 1 RAW DATA INTEGRITY")
print("=" * 90)

print("\n1. DATASET SHAPES")
print("-" * 90)
print(f"Schedule:       {schedule_df.shape}")
print(f"Race results:   {race_results_df.shape}")
print(f"Qualifying:     {qualifying_results_df.shape}")
print(f"Circuit lookup: {circuit_lookup_df.shape}")

print("\n2. UNIQUE RACES")
print("-" * 90)

schedule_races_count = schedule_df[["season", "round"]].drop_duplicates().shape[0]
results_races_count = race_results_df[["season", "round"]].drop_duplicates().shape[0]
quali_races_count = qualifying_results_df[["season", "round"]].drop_duplicates().shape[0]

print("Schedule races:   ", schedule_races_count)
print("Race result races:", results_races_count)
print("Qualifying races: ", quali_races_count)

print("\n3. UNIQUE ENTITIES")
print("-" * 90)
print("Drivers:", race_results_df["driverId"].nunique())
print("Constructors:", race_results_df["constructorId"].nunique())
print("Circuits:", schedule_df["circuitId"].nunique())
print("Countries:", schedule_df["country"].nunique())

print("\n4. DUPLICATES")
print("-" * 90)

driver_dupes = race_results_df.duplicated(["season", "round", "driverId"]).sum()
quali_dupes = qualifying_results_df.duplicated(["season", "round", "driverId"]).sum()
schedule_dupes = schedule_df.duplicated(["season", "round"]).sum()

print("Duplicate schedule races:", schedule_dupes)
print("Duplicate race-result rows:", driver_dupes)
print("Duplicate qualifying rows:", quali_dupes)

print("\n5. REQUIRED COLUMN MISSINGNESS")
print("-" * 90)

required_results = [
    "season",
    "round",
    "driverId",
    "constructorId",
    "position",
    "grid",
    "points"
]

for col in required_results:
    print(f"{col:<18}{race_results_df[col].isna().sum()}")

print("\n6. GRID SIZE")
print("-" * 90)

drivers_per_race = race_results_df.groupby(["season", "round"]).size()
print(drivers_per_race.describe())

print("\n7. CACHE")
print("-" * 90)

race_cache = len(list(RACE_CACHE.glob("*.pkl")))
quali_cache = len(list(QUALI_CACHE.glob("*.pkl")))

print("Race cache files:", race_cache)
print("Qualifying cache files:", quali_cache)

print("\n8. DATA RANGE")
print("-" * 90)

print("First schedule race:")
display(
    schedule_df
    .sort_values(["season", "round"])
    .head(1)[["season", "round", "raceName"]]
)

print("Last schedule race:")
display(
    schedule_df
    .sort_values(["season", "round"])
    .tail(1)[["season", "round", "raceName"]]
)

print("\n9. RANDOM SAMPLE")
print("-" * 90)

display(
    race_results_df.sample(10, random_state=42)[[
        "season",
        "round",
        "raceName",
        "driverId",
        "constructorId",
        "position",
        "grid",
        "points"
    ]]
)

print("\n10. HISTORICAL SANITY CHECKS")
print("-" * 90)

known_checks = [
    (2010, "Bahrain Grand Prix"),
    (2014, "Australian Grand Prix"),
    (2020, "British Grand Prix"),
    (2024, "Monaco Grand Prix"),
]

for season, race_name in known_checks:
    row = schedule_df[
        (schedule_df["season"] == season) &
        (schedule_df["raceName"] == race_name)
    ]

    display(row[["season", "round", "raceName", "circuitName"]])

print("\n11. FINAL VERDICT")
print("-" * 90)

checks = [
    schedule_races_count == results_races_count,
    schedule_races_count == quali_races_count,
    driver_dupes == 0,
    quali_dupes == 0,
    schedule_dupes == 0,
    race_cache == schedule_races_count,
    quali_cache == schedule_races_count,
]

if all(checks):
    print("PASS — Notebook 1 passed all integrity checks.")
else:
    print("REVIEW REQUIRED")

SUPERCHECK — NOTEBOOK 1 RAW DATA INTEGRITY

1. DATASET SHAPES
------------------------------------------------------------------------------------------
Schedule:       (329, 23)
Race results:   (6911, 37)
Qualifying:     (6891, 28)
Circuit lookup: (329, 9)

2. UNIQUE RACES
------------------------------------------------------------------------------------------
Schedule races:    329
Race result races: 329
Qualifying races:  329

3. UNIQUE ENTITIES
------------------------------------------------------------------------------------------
Drivers: 83
Constructors: 23
Circuits: 35
Countries: 29

4. DUPLICATES
------------------------------------------------------------------------------------------
Duplicate schedule races: 0
Duplicate race-result rows: 0
Duplicate qualifying rows: 0

5. REQUIRED COLUMN MISSINGNESS
------------------------------------------------------------------------------------------
season            0
round             0
driverId          0
constructorId     0
po

,season,round,raceName
0,2010,1,Bahrain Grand Prix


Last schedule race:


,season,round,raceName
328,2025,24,Abu Dhabi Grand Prix



9. RANDOM SAMPLE
------------------------------------------------------------------------------------------


...

,season,round,raceName,circuitName
0,2010,1,Bahrain Grand Prix,Bahrain International Circuit


,season,round,raceName,circuitName
77,2014,1,Australian Grand Prix,Albert Park Grand Prix Circuit


,season,round,raceName,circuitName
201,2020,4,British Grand Prix,Silverstone Circuit


,season,round,raceName,circuitName
288,2024,8,Monaco Grand Prix,Circuit de Monaco



11. FINAL VERDICT
------------------------------------------------------------------------------------------
PASS — Notebook 1 passed all integrity checks.


### Save

In [15]:
# ==============================================================================
# SAVE FINAL DATASETS
# ==============================================================================

race_results_df.to_csv(RAW / "race_results.csv", index=False)
qualifying_results_df.to_csv(RAW / "qualifying_results.csv", index=False)
schedule_df.to_csv(RAW / "race_schedule.csv", index=False)
circuit_lookup_df.to_csv(RAW / "circuit_lookup.csv", index=False)

print("Saved:")
print(" - race_results.csv")
print(" - qualifying_results.csv")
print(" - race_schedule.csv")
print(" - circuit_lookup.csv")

Saved:
 - race_results.csv
 - qualifying_results.csv
 - race_schedule.csv
 - circuit_lookup.csv


### Final Validation

In [16]:
# ==============================================================================
# VERIFY SAVED FILES
# ==============================================================================

for file in [
    "race_results.csv",
    "qualifying_results.csv",
    "race_schedule.csv",
    "circuit_lookup.csv",
]:
    path = RAW / file
    print(f"{file:25} Exists: {path.exists()}   Size: {path.stat().st_size:,} bytes")

race_results.csv          Exists: True   Size: 2,523,367 bytes
qualifying_results.csv    Exists: True   Size: 2,340,170 bytes
race_schedule.csv         Exists: True   Size: 96,084 bytes
circuit_lookup.csv        Exists: True   Size: 30,642 bytes
